# Switched-Array Radio Direction-Finding System

This notebook implements the complete hardware radio direction-finding system.

The system uses a permanent reference antenna connected to Pluto RX1 and a
set of antennas that are sequentially switched into Pluto RX2. An STM32
controls the RF switch, while Julia coordinates the switch states, acquires
the two receive channels, processes the I/Q data and estimates the direction
of the received signal.

The implementation is kept configurable so that the number of switched
antennas, operating frequency and array geometry can be changed without
restructuring the processing code.

In [ ]:
# Python environment used by pyadi-iio

pluto_python = raw"C:\Users\thoba\Documents\PlutoPythonTest\.venv\Scripts\python.exe"

ENV["JULIA_CONDAPKG_BACKEND"] = "Null"
ENV["JULIA_PYTHONCALL_EXE"] = pluto_python

In [ ]:
# Packages

using PythonCall
using Statistics
using FFTW
using LinearAlgebra
using Plots
using LibSerialPort
using IJulia

plotly()

adi = pyimport("adi")
subprocess = pyimport("subprocess")

In [ ]:
# ============================================================
# RDF system settings
# ============================================================

# Pluto receiver
sample_rate = 6_000_000
centre_frequency = 867_800_000
rf_bandwidth = 5_000_000
rx_buffer_size = 32768
rx_gain = 20.0

# Array
num_switched_antennas = 4

switched_antennas = [
    "A2",
    "A3",
    "A4",
    "A5"
]

# STM32
stm_port = "COM8"
stm_baud = 115200

# Number of captures discarded after switching
discard_buffers = 5

println("RDF settings loaded")

## Hardware Connections

In [ ]:
# Connect to the STM32 over the USB serial port

stm = LibSerialPort.open(stm_port, stm_baud)

println("STM32 connected on ", stm_port)

In [ ]:
# Find the Pluto Ethernet address

result = subprocess.run(
    ["iio_info", "-s"],
    capture_output = true,
    text = true
)

output = pyconvert(String, result.stdout)

pluto_ip = nothing

for line in split(output, '\n')

    if occursin("Pluto", line)

        for match in eachmatch(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", line)

            ip = match.match

            # Ignore the Pluto USB-network address
            if ip != "192.168.2.1"
                pluto_ip = ip
                break
            end
        end
    end
end

pluto_ip === nothing &&
    error("Pluto Ethernet connection not found.")

pluto_uri = "ip:" * pluto_ip

sdr = adi.ad9361(uri = pluto_uri)

println("Pluto connected at ", pluto_ip)

In [ ]:
# ============================================================
# Configure Pluto for RDF measurements
# ============================================================

function configure_rdf_receiver(sdr)

    builtins = pyimport("builtins")

    # Use both Pluto receive channels
    sdr.rx_enabled_channels = builtins.list((0, 1))

    # Receiver settings
    sdr.sample_rate = sample_rate
    sdr.rx_rf_bandwidth = rf_bandwidth
    sdr.rx_lo = centre_frequency
    sdr.rx_buffer_size = rx_buffer_size

    # Manual gain on both channels
    sdr.gain_control_mode_chan0 = "manual"
    sdr.gain_control_mode_chan1 = "manual"

    sdr.rx_hardwaregain_chan0 = rx_gain
    sdr.rx_hardwaregain_chan1 = rx_gain

    # Clear old samples
    for _ in 1:discard_buffers
        sdr.rx()
    end

    println("Pluto configured for RDF")
    println("Sample rate = ", sdr.sample_rate / 1e6, " MS/s")
    println("RX LO = ", sdr.rx_lo / 1e6, " MHz")
    println("RF bandwidth = ", sdr.rx_rf_bandwidth / 1e6, " MHz")
    println("RX gain = ", rx_gain, " dB")

    return nothing
end

In [ ]:
configure_rdf_receiver(sdr)

## RF Switch Control

In [ ]:
# ============================================================
# RF switch control
# ============================================================

function select_path(stm, path)

    if path < 1 || path > num_switched_antennas
        error("Invalid RF path: $path")
    end

    # Send RF path number to STM32
    write(stm, string(path))

    # Wait for confirmation
    response = strip(readline(stm))

    expected_response = "READY $path"

    if response != expected_response
        error(
            "Expected '$expected_response', received '$response'"
        )
    end

    return nothing
end

In [ ]:
# ============================================================
# Detect RF carrier from RX1
# ============================================================

function detect_vco(rx1, centre_frequency, sample_rate)

    # Remove DC component
    x = rx1 .- mean(rx1)

    N = length(x)
    n = 0:N-1

    # Hann window
    window = 0.5 .- 0.5 .* cos.(2π .* n ./ (N - 1))

    # FFT
    X = fftshift(fft(x .* window))
    power = abs2.(X)

    # Frequency axis
    freq = centre_frequency .+ (-N÷2:N÷2-1) .* (sample_rate / N)

    # Search around the expected 868 MHz source
    search = (freq .>= 867.8e6) .& (freq .<= 868.8e6)

    search_indices = findall(search)

    peak_index = search_indices[argmax(power[search])]

    f_vco = freq[peak_index]

    # Baseband frequency relative to Pluto LO
    f_bb = f_vco - centre_frequency

    return f_vco, f_bb, freq, power
end

In [ ]:
# Capture both receiver channels
rx_data = sdr.rx()

rx1 = pyconvert(Vector{ComplexF64}, rx_data[0])

# Detect transmitter
f_vco, f_bb, freq, power = detect_vco(rx1, centre_frequency, sample_rate)

println(
    "Detected VCO = ",
    round(f_vco / 1e6, digits = 6),
    " MHz"
)

println(
    "Baseband frequency = ",
    round(f_bb / 1e3, digits = 2),
    " kHz"
)

In [ ]:
# ============================================================
# Cell 3 — Single VCO measurement
# ============================================================

# Read Pluto settings
fc = pyconvert(Float64, sdr.rx_lo)
fs = pyconvert(Float64, sdr.sample_rate)

# Discard one old capture
sdr.rx()

# Capture both RX channels
rx_data = sdr.rx()

# Use A1 / RX1 to detect the VCO
rx1 = pyconvert(Vector{ComplexF64}, rx_data[0])

# Detect VCO
f_vco, f_bb, freq, power = detect_vco(rx1, fc, fs)

println(
    "Detected VCO = ",
    round(f_vco / 1e6, digits=6),
    " MHz"
)

println(
    "Baseband frequency = ",
    round(f_bb / 1e3, digits=2),
    " kHz"
)

# Convert power to dB for plotting
power_db = 10 .* log10.(power .+ 1e-20)

plot(
    freq ./ 1e6,
    power_db,
    xlabel = "RF Frequency (MHz)",
    ylabel = "Relative Power (dB)",
    title = "VCO Spectrum",
    legend = false,
    xlim = (867.8, 868.8)
)

vline!(
    [f_vco / 1e6],
    linestyle = :dash
)

In [ ]:
# ============================================================
# Live VCO spectrum
# ============================================================

for k in 1:50

    # Read Pluto settings
    fc = pyconvert(Float64, sdr.rx_lo)
    fs = pyconvert(Float64, sdr.sample_rate)

    # Capture both channels
    rx_data = sdr.rx()

    # Use A1 / RX1
    rx1 = pyconvert(Vector{ComplexF64}, rx_data[0])

    # Detect VCO
    f_vco, f_bb, freq, power = detect_vco(rx1, fc, fs)

    power_db = 10 .* log10.(power .+ 1e-20)

    # Plot spectrum
    p = plot(
        freq ./ 1e6,
        power_db,
        xlabel = "RF Frequency (MHz)",
        ylabel = "Relative Power (dB)",
        title =
            "Live VCO = $(round(f_vco/1e6, digits=4)) MHz",
        legend = false,
        xlim = (867.8, 868.8)
    )

    vline!(
        p,
        [f_vco / 1e6],
        linestyle = :dash
    )

    IJulia.clear_output(true)
    display(p)

    sleep(0.2)
end

## Direction Finding

In [ ]:
# ============================================================
# Fixed hardware phase calibration
# ============================================================

hardware_phase_deg = [
    20.211,   # A1 - A2 / RF1
    21.669,   # A1 - A3 / RF2
    21.460,   # A1 - A4 / RF3
    21.116    # A1 - A5 / RF4
]

In [ ]:
# ============================================================
# Measure all four hardware-corrected antenna phases
# ============================================================

function measure_array_phases(
    sdr,
    stm,
    f_bb;
    num_measurements = 20
)

    raw_phase_deg = zeros(4)
    corrected_phase_deg = zeros(4)
    phase_std_deg = zeros(4)
    max_deviation_deg = zeros(4)

    for path in 1:4

        # Select A2, A3, A4 or A5
        select_path(stm, path)

        # Allow switched path to settle
        for _ in 1:discard_buffers
            sdr.rx()
        end

        phase_measurements_deg = zeros(num_measurements)

        for measurement in 1:num_measurements

            rx_data = sdr.rx()

            rx1 = pyconvert(Vector{ComplexF64}, rx_data[0])

            rx2 = pyconvert(Vector{ComplexF64}, rx_data[1])

            N = rx_buffer_size
            n = 0:N-1

            # Hann window
            window = 0.5 .- 0.5 .* cos.(2π .* n ./ (N - 1))

            # Shift detected carrier to DC
            reference_tone = exp.(-1im .* 2π .* f_bb .* n ./ sample_rate)
            
            weights = window .* reference_tone

            V1 = sum(rx1 .* weights)

            V2 = sum(rx2 .* weights)

            # Phase convention: A1 - An
            phase_measurements_deg[measurement] = rad2deg(angle(V1 * conj(V2)))
        end

        # Circular mean
        raw_phase_deg[path] = rad2deg(angle(mean(exp.(1im .* deg2rad.(phase_measurements_deg)))))

        # Measurement spread
        phase_errors_deg = rad2deg.(angle.(exp.(1im .* deg2rad.(phase_measurements_deg .- raw_phase_deg[path]))))

        phase_std_deg[path] = std(phase_errors_deg)

        max_deviation_deg[path] = maximum(abs.(phase_errors_deg))

        # Hardware phase correction
        corrected_phase_deg[path] = rad2deg(angle(exp(1im * deg2rad(raw_phase_deg[path] - hardware_phase_deg[path]))))

        println(
            "A1 - A$(path + 1)",
            "   corrected = ",
            round(
                corrected_phase_deg[path],
                digits = 2
            ),
            "°   std = ",
            round(
                phase_std_deg[path],
                digits = 2
            ),
            "°"
        )
    end

    return (
        raw_phase_deg,
        corrected_phase_deg,
        phase_std_deg,
        max_deviation_deg
    )
end

In [ ]:
# ============================================================
# Fixed open-space array calibration — 15° spacing
# ============================================================

cal_angles_deg = collect(0:15:345)

cal_measured_deg = [
     70.13  -144.39  -155.63    73.59
     49.16  -176.41  -149.16    89.29
      6.40   135.33  -135.90   104.81
    -33.73    79.90  -157.58   110.54
    -54.61    47.33  -175.82   125.15
    -73.88     9.41   118.36   125.76
    -94.58   -48.06    71.20   120.03
    -99.06   -98.82    15.16   116.36
   -101.25  -137.93   -36.84    82.76
   -113.11  -176.27  -109.38    19.38
   -112.18   162.21  -147.27   -19.94
   -103.44   154.69   163.96   -58.66
    -84.39   158.88   138.45   -83.37
    -72.55   169.70   118.74  -116.35
    -38.24  -149.18   130.31  -137.70
     15.34   -90.86   167.51  -132.91
     50.85   -47.36  -157.99  -138.23
     85.14    -3.58  -115.70  -133.66
    107.04    50.02   -57.21  -119.56
    132.17   106.57   -11.45   -90.73
    134.48   158.97    32.48   -51.32
    128.71  -161.67    74.66    -8.47
    118.96  -137.32   119.40    22.10
    100.68  -135.36   170.05    49.00
]

In [ ]:
# ============================================================
# Ideal UCA phase signatures at the 24 calibration angles
# ============================================================

c = 3e8
f0 = 868.1e6
λ = c / f0

Nant = 5
Rcircle = 0.09

A = zeros(Nant, 3)

for n in 1:Nant

    antenna_angle = 2π * (n - 1) / Nant

    A[n, 1] = Rcircle * cos(antenna_angle)

    A[n, 2] = Rcircle * sin(antenna_angle)
end

cal_ideal_deg = zeros(24, 4)

for (i, theta_deg) in enumerate(cal_angles_deg)

    theta_rad = deg2rad(theta_deg)

    u = [
        cos(theta_rad),
        sin(theta_rad),
        0
    ]

    for n in 2:5

        path_difference = dot(u, A[1, :] - A[n, :])

        phase_deg = rad2deg(2π * path_difference / λ)

        cal_ideal_deg[i, n - 1] = rad2deg(angle(exp(1im * deg2rad(phase_deg))))
    end
end

println(
    "Ideal phase matrix size = ",
    size(cal_ideal_deg)
)

In [ ]:
# ============================================================
# Array phase correction at the 24 calibration angles
# ============================================================

array_correction_deg = zeros(24, 4)

for i in 1:24
    for j in 1:4

        difference = cal_measured_deg[i, j] - cal_ideal_deg[i, j]

        array_correction_deg[i, j] = rad2deg(angle(exp(1im * deg2rad(difference))))
    end
end

println(
    "Array correction matrix size = ",
    size(array_correction_deg)
)

In [ ]:
# ============================================================
# Interpolate array correction from 0° to 359°
# ============================================================

scan_angles_deg = collect(0:359)

array_correction_interp_deg =  zeros(360, 4)

for theta in scan_angles_deg

    # Find the 15° calibration interval
    lower_index = floor(Int, theta / 15) + 1

    # Wrap 345° -> 0°
    upper_index = lower_index == 24 ? 1 : lower_index + 1

    lower_angle = cal_angles_deg[lower_index]

    fraction = (theta - lower_angle) / 15

    for j in 1:4

        c1 = array_correction_deg[lower_index, j]

        c2 = array_correction_deg[upper_index, j]

        # Shortest angular change
        correction_change = rad2deg(angle(exp(1im * deg2rad(c2 - c1))))

        interpolated = c1 + fraction * correction_change

        array_correction_interp_deg[theta + 1, j] =rad2deg(angle(exp(1im * deg2rad(interpolated))))
    end
end

println(
    "Interpolated correction matrix size = ",
    size(array_correction_interp_deg)
)

In [ ]:
# ============================================================
# Final calibrated phase lookup: 0° to 359°
# ============================================================

calibrated_phase_deg = zeros(360, 4)

for theta in scan_angles_deg

    theta_rad = deg2rad(theta)

    u = [
        cos(theta_rad),
        sin(theta_rad),
        0
    ]

    for n in 2:5

        # Ideal geometrical phase
        path_difference = dot(u, A[1, :] - A[n, :])

        ideal_phase_deg = rad2deg(2π * path_difference / λ)

        # Ideal phase + interpolated array correction
        calibrated_phase_deg[theta + 1, n - 1] =
            rad2deg(angle(exp(1im * deg2rad(ideal_phase_deg + array_correction_interp_deg[theta + 1, n - 1]))))
    end
end

println(
    "Calibrated lookup matrix size = ",
    size(calibrated_phase_deg)
)

In [ ]:
# ============================================================
# Estimate direction of arrival
# ============================================================

function estimate_doa(
    measured_phase_deg,
    calibrated_phase_deg
)

    rmse_deg = zeros(360)

    for theta in 0:359

        phase_error_deg = zeros(4)

        for j in 1:4

            difference = measured_phase_deg[j] - calibrated_phase_deg[theta + 1, j]

            phase_error_deg[j] = rad2deg(angle(exp(1im * deg2rad(difference))))
        end

        rmse_deg[theta + 1] = sqrt(mean(phase_error_deg .^ 2))
    end

    estimated_angle = argmin(rmse_deg) - 1

    minimum_rmse = minimum(rmse_deg)

    return estimated_angle, minimum_rmse, rmse_deg
end

In [ ]:
# ============================================================
# Complete live RDF measurement
# ============================================================

function estimate_live_direction(sdr, stm)

    # --------------------------------------------------------
    # 1. Capture signal and detect carrier
    # --------------------------------------------------------

    rx_data = sdr.rx()

    rx1 = pyconvert(
        Vector{ComplexF64},
        rx_data[0]
    )

    f_vco, f_bb, freq, power =
        detect_vco(
            rx1,
            centre_frequency,
            sample_rate
        )

    # --------------------------------------------------------
    # 2. Measure antenna phase differences
    # --------------------------------------------------------

    raw_phase_deg,
    measured_phase_deg,
    phase_std_deg,
    max_deviation_deg =
        measure_array_phases(
            sdr,
            stm,
            f_bb
        )

    # --------------------------------------------------------
    # 3. Estimate direction
    # --------------------------------------------------------

    estimated_angle,
    minimum_rmse,
    rmse_deg =
        estimate_doa(
            measured_phase_deg,
            calibrated_phase_deg
        )

    # --------------------------------------------------------
    # 4. Display result
    # --------------------------------------------------------

    println()

    println(
        "Carrier = ",
        round(f_vco / 1e6, digits = 6),
        " MHz"
    )

    println(
        "Estimated DoA = ",
        estimated_angle,
        "°"
    )

    println(
        "Phase RMSE = ",
        round(minimum_rmse, digits = 2),
        "°"
    )

    # --------------------------------------------------------
    # 5. Return RDF + spectrum data
    # --------------------------------------------------------

    return (
        estimated_angle,
        minimum_rmse,
        measured_phase_deg,
        phase_std_deg,
        f_vco,
        freq,
        power
    )
end

In [ ]:
spectrum_freq_mhz,
spectrum_power_db =
    prepare_gui_spectrum(
        freq,
        power
    )

println(
    "Spectrum points = ",
    length(spectrum_freq_mhz)
)

In [ ]:
estimated_angle,
minimum_rmse,
measured_phase_deg,
phase_std_deg,
f_vco,
freq,
power =
    estimate_live_direction(sdr, stm);

# Live RDF Display

In [ ]:
# ============================================================
# Live RDF display
# ============================================================

using GLMakie

In [ ]:
# ============================================================
# Live RDF display
# ============================================================

Makie.inline!(false)
GLMakie.activate!()

println("GLMakie ready")

In [ ]:
# ============================================================
# Prepare spectrum data for GUI
# ============================================================

function prepare_gui_spectrum(freq, power)

    spectrum_mask =
        (freq .>= 867.8e6) .&
        (freq .<= 868.4e6)

    spectrum_freq_mhz =
        freq[spectrum_mask] ./ 1e6

    spectrum_power =
        power[spectrum_mask]

    spectrum_power_db =
        10 .* log10.(
            spectrum_power ./
            maximum(spectrum_power)
        )

    return (
        spectrum_freq_mhz,
        spectrum_power_db
    )
end

In [ ]:
spectrum_freq_mhz, spectrum_power_db =
    prepare_gui_spectrum(freq, power)

In [ ]:
# ============================================================
# Live GUI observables
# ============================================================

# ------------------------------------------------------------
# Helper: build filled DoA pointer from an angle
# ------------------------------------------------------------

function make_pointer(angle_deg)

    angle_rad =
        deg2rad(angle_deg)

    pointer_length = 0.80
    pointer_width = 0.055

    ux = cos(angle_rad)
    uy = sin(angle_rad)

    px = -sin(angle_rad)
    py =  cos(angle_rad)

    return GLMakie.Point2f[

        GLMakie.Point2f(
            -0.03 * ux,
            -0.03 * uy
        ),

        GLMakie.Point2f(
            0.10 * ux + pointer_width * px,
            0.10 * uy + pointer_width * py
        ),

        GLMakie.Point2f(
            pointer_length * ux,
            pointer_length * uy
        ),

        GLMakie.Point2f(
            0.10 * ux - pointer_width * px,
            0.10 * uy - pointer_width * py
        )
    ]
end

# ------------------------------------------------------------
# Live measured values
# ------------------------------------------------------------

doa_obs =
    GLMakie.Observable(
        Float64(estimated_angle)
    )

carrier_obs =
    GLMakie.Observable(
        Float64(f_vco / 1e6)
    )

rmse_obs =
    GLMakie.Observable(
        Float64(minimum_rmse)
    )

spectrum_freq_obs =
    GLMakie.Observable(
        spectrum_freq_mhz
    )

spectrum_power_obs =
    GLMakie.Observable(
        spectrum_power_db
    )

# ------------------------------------------------------------
# Values derived from the measurements
# ------------------------------------------------------------

pointer_obs =
    GLMakie.lift(doa_obs) do angle_deg
        make_pointer(angle_deg)
    end

doa_text_obs =
    GLMakie.lift(doa_obs) do angle_deg
        "$(round(Int, angle_deg))°"
    end

carrier_text_obs =
    GLMakie.lift(carrier_obs) do carrier_mhz
        "$(round(carrier_mhz, digits = 3)) MHz"
    end

rmse_text_obs =
    GLMakie.lift(rmse_obs) do rmse
        "$(round(rmse, digits = 2))°"
    end

println("Live GUI observables created")

In [ ]:
# ============================================================
# Final live-ready RDF GUI
# ============================================================

GLMakie.closeall()

# ------------------------------------------------------------
# Colours
# ------------------------------------------------------------

background_colour =
    GLMakie.RGBf(0.04, 0.06, 0.04)

panel_colour =
    GLMakie.RGBf(0.06, 0.09, 0.06)

live_colour =
    GLMakie.RGBf(0.00, 1.00, 0.25)

structure_colour =
    GLMakie.RGBf(0.22, 0.34, 0.24)

text_colour =
    GLMakie.RGBf(0.74, 0.79, 0.74)

dim_text_colour =
    GLMakie.RGBf(0.45, 0.52, 0.45)

transparent_colour =
    GLMakie.RGBAf(0.0, 0.0, 0.0, 0.0)

# ------------------------------------------------------------
# Carrier marker for spectrum
# ------------------------------------------------------------

carrier_marker_obs =
    GLMakie.lift(carrier_obs) do carrier_mhz
        [carrier_mhz]
    end

# ------------------------------------------------------------
# Main figure
# ------------------------------------------------------------

fig_gui = GLMakie.Figure(
    size = (1500, 900),
    backgroundcolor = background_colour
)

# ============================================================
# FULL-WIDTH HEADER
# ============================================================

GLMakie.Box(
    fig_gui[1:2, 1:2],
    color = panel_colour,
    strokecolor = structure_colour,
    strokewidth = 1.5
)

GLMakie.Label(
    fig_gui[1, 1:2],
    "RADIO DIRECTION FINDER",
    fontsize = 29,
    color = text_colour,
    font = :bold
)

GLMakie.Label(
    fig_gui[2, 1:2],
    "SOFTWARE-DEFINED RADIO DIRECTION ESTIMATION",
    fontsize = 13,
    color = dim_text_colour
)

# ============================================================
# MAIN DISPLAY SECTION
# ============================================================

content_grid =
    fig_gui[3, 1:2] =
    GLMakie.GridLayout()

GLMakie.colgap!(
    content_grid,
    18
)

GLMakie.rowgap!(
    content_grid,
    0
)

# ============================================================
# BEARING SUB-GRID
# ============================================================

bearing_grid =
    content_grid[1, 1] =
    GLMakie.GridLayout()

GLMakie.rowgap!(
    bearing_grid,
    6
)

GLMakie.Label(
    bearing_grid[1, 1],
    "BEARING",
    fontsize = 17,
    color = text_colour,
    font = :bold
)

# ============================================================
# SPECTRUM SUB-GRID
# ============================================================

spectrum_grid =
    content_grid[1, 3] =
    GLMakie.GridLayout()

GLMakie.rowgap!(
    spectrum_grid,
    6
)

GLMakie.Label(
    spectrum_grid[1, 1],
    "RF SPECTRUM",
    fontsize = 17,
    color = text_colour,
    font = :bold
)

# ------------------------------------------------------------
# Centre divider
# ------------------------------------------------------------

GLMakie.Box(
    content_grid[1, 2],
    color = structure_colour,
    strokewidth = 0
)

# ============================================================
# LEFT: BEARING DISPLAY
# ============================================================

ax_bearing = GLMakie.Axis(
    bearing_grid[2, 1],

    width = 610,
    height = 500,

    aspect = GLMakie.DataAspect(),

    limits = (
        -1.18,
         1.18,
        -1.18,
         1.18
    ),

    backgroundcolor = background_colour
)

GLMakie.hidedecorations!(ax_bearing)
GLMakie.hidespines!(ax_bearing)

# ------------------------------------------------------------
# Compass circle
# ------------------------------------------------------------

circle_angle =
    range(
        0,
        2π,
        length = 361
    )

GLMakie.lines!(
    ax_bearing,
    cos.(circle_angle),
    sin.(circle_angle),
    color = structure_colour,
    linewidth = 2.5
)

# ------------------------------------------------------------
# Tick marks
# ------------------------------------------------------------

for angle_deg in 0:10:350

    angle_rad =
        deg2rad(angle_deg)

    inner_radius =
        angle_deg % 30 == 0 ?
        0.85 :
        0.93

    tick_width =
        angle_deg % 30 == 0 ?
        2.6 :
        1.0

    GLMakie.lines!(
        ax_bearing,

        [
            inner_radius * cos(angle_rad),
            cos(angle_rad)
        ],

        [
            inner_radius * sin(angle_rad),
            sin(angle_rad)
        ],

        color = structure_colour,
        linewidth = tick_width
    )
end

# ------------------------------------------------------------
# Main angle labels
# ------------------------------------------------------------

for (label, x, y) in [

    ("0°",    1.10,  0.00),
    ("90°",   0.00,  1.10),
    ("180°", -1.10,  0.00),
    ("270°",  0.00, -1.10)

]

    GLMakie.text!(
        ax_bearing,
        label,

        position = (x, y),

        align = (
            :center,
            :center
        ),

        fontsize = 18,
        color = text_colour
    )
end

# ------------------------------------------------------------
# LIVE filled DoA pointer
# ------------------------------------------------------------

GLMakie.poly!(
    ax_bearing,
    pointer_obs,
    color = live_colour
)

# ============================================================
# RIGHT: LIVE RF SPECTRUM
# ============================================================

ax_spectrum = GLMakie.Axis(
    spectrum_grid[2, 1],

    width = 610,
    height = 500,

    xlabel = "Frequency (MHz)",
    ylabel = "Relative power (dB)",

    backgroundcolor = background_colour,

    xlabelcolor = text_colour,
    ylabelcolor = text_colour,

    xticklabelcolor = text_colour,
    yticklabelcolor = text_colour,

    xgridcolor =
        GLMakie.RGBAf(
            0.15,
            0.23,
            0.16,
            0.45
        ),

    ygridcolor =
        GLMakie.RGBAf(
            0.15,
            0.23,
            0.16,
            0.45
        ),

    topspinecolor = structure_colour,
    bottomspinecolor = structure_colour,
    leftspinecolor = structure_colour,
    rightspinecolor = structure_colour
)

# Live spectrum trace
GLMakie.lines!(
    ax_spectrum,
    spectrum_freq_obs,
    spectrum_power_obs,
    color = live_colour,
    linewidth = 1.5
)

# Live carrier marker
GLMakie.vlines!(
    ax_spectrum,
    carrier_marker_obs,
    color = live_colour,
    linewidth = 2
)

GLMakie.ylims!(
    ax_spectrum,
    -110,
    5
)

# Keep the reduced frequency span
GLMakie.xlims!(
    ax_spectrum,
    carrier_obs[] - 0.15,
    carrier_obs[] + 0.15
)

# ============================================================
# SIZE MAIN DISPLAY COLUMNS
# ============================================================

GLMakie.colsize!(
    content_grid,
    1,
    Makie.Relative(0.5)
)

GLMakie.colsize!(
    content_grid,
    2,
    Makie.Fixed(2)
)

GLMakie.colsize!(
    content_grid,
    3,
    Makie.Relative(0.5)
)

# ============================================================
# OUTER FRAME AROUND BOTH DISPLAYS
# ============================================================

GLMakie.Box(
    content_grid[1, 1:3],
    color = transparent_colour,
    strokecolor = structure_colour,
    strokewidth = 1.5
)

# ============================================================
# FULL-WIDTH BOTTOM TABLE
# ============================================================

status_grid =
    fig_gui[4, 1:2] =
    GLMakie.GridLayout()

GLMakie.colgap!(
    status_grid,
    0
)

GLMakie.rowgap!(
    status_grid,
    0
)

headings = [
    "DOA",
    "CARRIER",
    "PHASE FIT RMSE"
]

value_observables = [
    doa_text_obs,
    carrier_text_obs,
    rmse_text_obs
]

content_cols = [
    1,
    3,
    5
]

separator_cols = [
    2,
    4
]

# ------------------------------------------------------------
# Outer status frame
# ------------------------------------------------------------

GLMakie.Box(
    status_grid[1:2, 1:5],
    color = panel_colour,
    strokecolor = structure_colour,
    strokewidth = 1.5
)

# ------------------------------------------------------------
# Headings + LIVE values
# ------------------------------------------------------------

for (i, col) in enumerate(content_cols)

    GLMakie.Label(
        status_grid[1, col],
        headings[i],
        fontsize = 12,
        color = dim_text_colour
    )

    GLMakie.Label(
        status_grid[2, col],
        value_observables[i],
        fontsize = 21,
        color = text_colour,
        font = :bold
    )
end

# ------------------------------------------------------------
# Internal separators
# ------------------------------------------------------------

for col in separator_cols

    GLMakie.Box(
        status_grid[1:2, col],
        color = structure_colour,
        strokewidth = 0
    )
end

# ------------------------------------------------------------
# Equal bottom columns
# ------------------------------------------------------------

for col in content_cols

    GLMakie.colsize!(
        status_grid,
        col,
        Makie.Relative(0.33)
    )
end

for col in separator_cols

    GLMakie.colsize!(
        status_grid,
        col,
        Makie.Fixed(2)
    )
end

# ============================================================
# MAIN FIGURE SPACING
# ============================================================

GLMakie.rowgap!(
    fig_gui.layout,
    12
)

GLMakie.colgap!(
    fig_gui.layout,
    0
)

# ============================================================
# OPEN GUI WINDOW
# ============================================================

screen_gui =
    GLMakie.Screen()

display(
    screen_gui,
    fig_gui
)

In [ ]:
# ============================================================
# Live RDF measurement and GUI update
# ============================================================

live_running = Ref(true)

live_task = @async begin

    while live_running[]

        try
            angle, rmse, phases, phase_std,
            carrier, freq, power =
                estimate_live_direction(sdr, stm)

            gui_freq, gui_power =
                prepare_gui_spectrum(freq, power)

            # Update GUI
            doa_obs[] = Float64(angle)
            carrier_obs[] = Float64(carrier / 1e6)
            rmse_obs[] = Float64(rmse)

            spectrum_freq_obs[] = gui_freq
            spectrum_power_obs[] = gui_power

            # Follow detected carrier
            GLMakie.xlims!(
                ax_spectrum,
                carrier / 1e6 - 0.15,
                carrier / 1e6 + 0.15
            )

        catch err
            @warn "Live RDF measurement failed" exception = (err, catch_backtrace())
        end

        sleep(0.1)
    end

    println("Live RDF stopped")

end

In [ ]:
live_running[] = false

## Optional Utility: VCO Source Characterisation

The VCO carrier frequency is not known exactly because it depends on the
tuning voltage. The Pluto is therefore swept across a wide frequency range
and the received spectrum is examined to locate the strongest RF carrier.

In [ ]:
# Frequency range to search for the VCO carrier

sweep_start = 650_000_000       # 650 MHz
sweep_stop  = 1_100_000_000     # 1100 MHz

sweep_sample_rate = 6_000_000   # 5 MS/s
sweep_bandwidth = 5_000_000     # 4 MHz
sweep_step = 5_000_000          # Move 4 MHz at a time

sweep_buffer_size = 4096

# Centre frequencies that the Pluto will visit during the sweep

sweep_frequencies = sweep_start : sweep_step : sweep_stop

In [ ]:
# Configure Pluto for the VCO frequency sweep

sweep_gain = 10.0     

pysetattr(sdr, "rx_enabled_channels", pylist([0]))
pysetattr(sdr, "sample_rate", sweep_sample_rate)
pysetattr(sdr, "rx_rf_bandwidth", sweep_bandwidth)
pysetattr(sdr, "rx_buffer_size", sweep_buffer_size)

pysetattr(sdr, "gain_control_mode_chan0", "manual")
pysetattr(sdr, "rx_hardwaregain_chan0", sweep_gain)

println("Sweep sample rate: ", sweep_sample_rate / 1e6, " MS/s")
println("Sweep bandwidth:   ", sweep_bandwidth / 1e6, " MHz")
println("Sweep RX gain:     ", sweep_gain, " dB")

In [ ]:
# Storage for the complete measured spectrum

all_frequencies = Float64[]
all_magnitudes = Float64[]


for centre_freq in sweep_frequencies
    
    # Tune Pluto to the next centre frequency
    pysetattr(sdr, "rx_lo", centre_freq)
    
    # Discard one capture after changing frequency
    # so that the next capture belongs to the new tuning position
    sdr.rx()
    
    # Capture IQ samples
    rx_data = sdr.rx()
    rx = pyconvert(Vector{ComplexF64}, rx_data)
    
    # Remove DC offset
    rx = rx .- mean(rx)
    
    # Calculate the spectrum
    N = length(rx)
    
    frequency_bins = -N÷2 : N÷2-1
    baseband_frequency = frequency_bins .* (sweep_sample_rate / N)
    
    spectrum = abs.(fftshift(fft(rx)))
    
    # Convert baseband frequency to actual RF frequency
    rf_frequency = centre_freq .+ baseband_frequency

    # Keep only the central 4 MHz of this capture
    keep = abs.(baseband_frequency) .<= sweep_bandwidth / 2

    append!(all_frequencies, rf_frequency[keep])
    append!(all_magnitudes, spectrum[keep])
end

println("Sweep complete")

In [ ]:
peak_index = argmax(all_magnitudes)

peak_frequency = all_frequencies[peak_index]
peak_magnitude = all_magnitudes[peak_index]

println("Strongest detected frequency: ", round(peak_frequency / 1e6, digits = 3), " MHz")

In [ ]:
# Plot the FFT magnitude across the complete sweep

plot(
    all_frequencies ./ 1e6,
    all_magnitudes,

    xlabel = "Frequency (MHz)",
    ylabel = "FFT Magnitude",
    title = "VCO Frequency Spectrum",

    legend = false,
    linewidth = 1
)

In [ ]:
# Restore the Pluto to the normal RDF receiver configuration
configure_rdf_receiver(sdr)